# Session 11 · Multi-provider calls
## One function, three providers (Groq, Gemini, Ollama), and the sampling controls

**How to use this notebook.** It has three parts.

| Part | What it is | Code? |
|---|---|---|
| **Part A · Read before you run** | Why this topic matters, the words you need, and how today's session is built | No code. Read it first (about 15 minutes). |
| **Part B · Hands-on** | Sections 1 to 7. Every section follows the same six steps (below). | Yes |
| **Part C · Wrap-up** | Summary, interview questions, troubleshooting | No |

Every section in Part B follows the same six steps, so you always know where you are:

1. **What and why** — what we are about to do and why it matters.
2. **The idea** — the concept, with an analogy and often a small diagram.
3. **What the next cell does** — a numbered walk-through *before* you see the code.
4. **The code** — run it.
5. **What you should see** — how to read the output, and what it means if yours is different.
6. **Check yourself** — one short question and a common mistake to avoid.

Boxes marked **▸ Read more** are optional. Click to open them when you want the deeper background; you can skip them in class.

**Before you start:** `git pull`, then `uv sync` (no new packages). Groq needs the key already in your `.env`; Gemini needs a free key (section 1). Ollama is optional. With no key, the notebook still runs top to bottom: model cells print `skipped (...)`.

---
# Part A · Read before you run

## A1. Why this session matters

### The problem, in one story
Imagine a bank whose assistant reads every customer complaint and sends it to one queue: **billing**, **technical**, or **general**. It works in the demo. Then it goes live, and three things happen in the first quarter.

1. **The provider has to change.** The risk team rules that complaints with account details must go to a model the bank hosts itself (the data-residency rule from Session 10). But the vendor's address and model name are written into fourteen files. A configuration change becomes three weeks of edits and re-testing.
2. **The same complaint gets two answers.** *"I upgraded my plan but I'm still charged the old price and the new features don't show."* On Monday it goes to *billing*; sent again on Tuesday, word for word, it goes to *technical*. The code did not change. The call used a high **temperature**, so the model sometimes picked a less likely answer. An auditor asks the team to reproduce Monday's decision, and they cannot.
3. **Replies stop mid-sentence.** A very small **max_tokens** was set to save money. Explanations are cut off, and with a reasoning model some replies are empty: the hidden thinking used the whole budget (Session 10).

None of these is a problem with the model. **They are problems with how the call was made.** That is today's topic.

### What you learn today, and what each one prevents

| You learn to... | It prevents... | The everyday name |
|---|---|---|
| Call any provider through **one function** (`ask()`), changing only `BASE_URL`, `API_KEY`, `MODEL` | Weeks of rewrites when a provider, price or policy changes | Swap the plug, not the appliance |
| Use the **system message** on purpose | A model that falls back to a chatty default personality | The job description |
| Set **temperature** low for decisions | The same ticket landing in different queues on different days | The randomness dial |
| Set **max_tokens** and **stop** deliberately | Cut-off or empty replies, and answers that ramble past the shape you need | The ceiling and the tripwire |
| **Verify** a model name live instead of trusting a written one | Code that breaks when a vendor retires a model name | Check, don't memorise |

### Where this is used in industry
- **Banking and insurance:** firms often use more than one provider, for example a hosted API for general text and a self-hosted model for data that must stay in-house. Provider-in-configuration makes that possible.
- **LLM gateways:** one routing layer in front of several providers (open-source libraries such as LiteLLM, or a cloud gateway) lets teams switch or fail over by configuration. Session 21 builds a small version with LangChain.
- **One request shape everywhere:** Groq, Google's Gemini endpoint, Ollama and self-hosting servers such as vLLM accept the OpenAI-compatible shape. Learn it once.
- **Low temperature for decisions:** complaint routing in banks, claims triage in insurance, ticket triage in telecom and retail; clinical-note summaries in healthcare, where the same note should give the same summary. Creative drafting uses higher values.

### Why interviewers care
These questions come up in AI-engineer and GenAI-developer interviews:
- *"How would you design an LLM application so you can switch providers without rewriting code?"*
- *"What does temperature do, and what would you set it to for a classifier?"*
- *"What is the difference between temperature and top_p?"*
- *"Is temperature 0 fully deterministic?"*
- *"What happens if max_tokens is too low, especially on a reasoning model?"*

By the end of today you can answer each one with a definition, an example from your own lab, and one honest limit. Part C has model answers.

<details><summary><b>▸ Read more: why does a "standard" request shape exist at all?</b></summary>

OpenAI's chat-completions API was one of the first widely used LLM APIs, so many tools were written for its shape (`model`, `messages`, sampling settings; a reply with `choices` and `usage`). Other vendors accept that shape so developers need not learn a new one. "Compatible" covers the common features; vendor-specific ones may differ, so teams still test each provider.
</details>

## A2. Where this session fits in the course

```
 S09  How LLMs work ───────────► tokens and next-token prediction: today's temperature acts on that
  │                              prediction, one token at a time
 S10  The model landscape ─────► which model for which job; reasoning models spend hidden tokens
  │                              (that is why max_tokens can give an empty answer today)
 S11  Multi-provider calls ────► TODAY: one ask() for Groq, Gemini, Ollama + temperature, max_tokens, stop
  │
 S12  Local and open models ───► Ollama in depth, the SAME client pointed at your laptop
  │
 S13  Multimodal + structured ─► images and JSON output, still the same messages list
  │
 S14  Prompt anatomy ──────────► what to write INSIDE the system and user messages
  │
 S15–S16  Reasoning, structure ► sampling several answers and voting; groq_client() from today is reused
  │
 S21  LangChain ───────────────► swap providers through a routing layer, without code changes
 S58–S60  Cloud AI platforms ──► the same idea at enterprise scale (Module 12)
```

**What you need from earlier sessions:**
- `uv sync` done and `uv run pytest tests/test_setup.py` green (Session 2).
- `.env` in the course folder with your Groq key in `API_KEY`; `BASE_URL` and `MODEL` as in `.env.example`. Do not change `MODEL`; the instructor pins it.
- Tokens and next-token prediction (Session 9); hidden reasoning tokens (Session 10).
- Optional: Ollama with `ollama pull llama3.2:3b` done (Session 2 homework).

## A3. The words you need today

Read these once now. Each term appears in bold the first time it is used in Part B.

---
**1. Provider**
- **Definition:** the company or program that runs a model and answers your requests. Today: Groq, Google (Gemini), and Ollama on your laptop.
- **Analogy:** a place. A provider is *where* the brain lives.
- **Example:** Groq at `https://api.groq.com/openai/v1`.
- **Often confused with:** the model. One provider offers many models, and one open-weight model can be offered by many providers.
- **Comes back in:** S12, S21, S58–S60.

---
**2. Model**
- **Definition:** the trained network that writes the answer, chosen by its name in the request.
- **Analogy:** the brain that answers.
- **Example:** `openai/gpt-oss-20b` on Groq (the course default), `llama3.2:3b` on Ollama.
- **Comes back in:** S10, S12, S15.

---
**3. The three settings: `BASE_URL`, `API_KEY`, `MODEL`**
- **Definition:** where to send the request (the address), who you are (a secret key), and which model should answer. They live in `.env`, never in code.
- **Analogy:** the address on the envelope, your PIN, and the name of the person you want to reach.
- **Example:** for Ollama: `http://localhost:11434/v1`, the word `ollama` (no real key needed), `llama3.2:3b`.
- **Often confused with:** settings you may share. Never paste a key in chat, code or a screenshot; if it leaks, revoke it and create a new one.
- **Comes back in:** S12, S22, S55, S60.

---
**4. OpenAI-compatible endpoint**
- **Definition:** an address that accepts the same request shape as OpenAI's chat-completions API and answers in the same shape, though another vendor runs it.
- **Analogy:** same language, different speaker.
- **Example:** Google's `https://generativelanguage.googleapis.com/v1beta/openai/` lets the `openai` Python library talk to Gemini.
- **Often confused with:** "uses OpenAI's models". Only the *format* is shared.
- **Comes back in:** S12, S21.

---
**5. Client**
- **Definition:** the Python object that holds one address and one key and sends requests for you: `OpenAI(base_url=..., api_key=...)`.
- **Analogy:** a phone with one number saved.
- **Example:** `groq_client()`, `gemini_client()`, `ollama_client()`: three phones of the same make.
- **Often confused with:** a connection. Creating a client contacts nobody; the first real request does.
- **Comes back in:** S12, S20.

---
**6. Provider-agnostic code**
- **Definition:** code that works unchanged with any provider, because only the three settings change.
- **Analogy:** swap the plug, not the appliance.
- **Example:** `ask()` is the same for Groq, Gemini and Ollama.
- **Often confused with:** "every provider gives the same answer". The *code* is the same; answers, speed and price differ.
- **Comes back in:** S21, S58–S60.

---
**7. Messages list and roles**
- **Definition:** the conversation you send: a list of dictionaries, each with a `role` and `content`. **system** sets behaviour, **user** is the person, **assistant** is the model's earlier replies.
- **Analogy:** the job description (system), the customer at the counter (user), the clerk's notes of what they already said (assistant).
- **Example:** `[{"role": "system", "content": "You are terse."}, {"role": "user", "content": "What is an EMI?"}]`
- **Often confused with:** a single prompt string. The system message stays the same on every request; the user message changes.
- **Comes back in:** every session; S14, S19 (chat memory), S20 (tool results), S34 onwards.

---
**8. Token usage**
- **Definition:** the counts returned with each reply: **prompt tokens** (what you sent), **completion tokens** (what it wrote), and the **total**. Price and rate limits are counted in tokens.
- **Analogy:** the itemised bill.
- **Often confused with:** words. Each provider has its own tokenizer (Session 9), so the same text can count differently.
- **Comes back in:** S15 (token meter), S19 (token budgets).

---
**9. temperature**
- **Definition:** a number (0 to roughly 2) that sets how much randomness goes into picking each next token. 0 is as close to "always pick the most likely token" as the provider allows.
- **Analogy:** the randomness dial.
- **Example:** classify a ticket at `0.0`; brainstorm slogans at `0.9`.
- **Often confused with:** creativity. Higher temperature means more varied choices, not better ideas, and sometimes less coherent ones.
- **Comes back in:** S15 (self-consistency), S34 (agents keep it low).

---
**10. top_p**
- **Definition:** another way to shape the same randomness: choose only among the most likely tokens whose probabilities add up to `top_p`.
- **Analogy:** a shortlist.
- **Often confused with:** something to set together with temperature. Teams usually change one and leave the other at its default; today's `ask()` does not expose it.

---
**11. max_tokens**
- **Definition:** a hard ceiling on the reply's length in tokens. The model is stopped when it reaches it, even mid-sentence.
- **Analogy:** the page limit, not the target length.
- **Example:** `max_tokens=15` cuts an explanation after a few words.
- **Often confused with:** "the length I want". On a reasoning model the hidden thinking counts too, so a low limit can leave nothing visible.
- **Comes back in:** S10, S15, S19.

---
**12. Stop sequence (`stop`)**
- **Definition:** a string that ends generation the moment it appears. The string itself is not included.
- **Analogy:** a tripwire for the output.
- **Example:** `stop=["\n"]` keeps only the first line.
- **Comes back in:** S13 and the structured-output labs.

---
**13. Drift**
- **Definition:** the same input getting different outputs on repeated runs with the same settings.
- **Analogy:** a coin flip hidden inside a decision.
- **Example:** `temperature=1.0  DRIFTED  ['billing', 'technical', 'billing']`
- **Often confused with:** a bug in your code. It is how sampling works; temperature controls it.
- **Comes back in:** S15 (voting over samples), S17 (evaluation), S52.

## A4. How this session is built

### The big picture

```
┌── COURSE REPOSITORY  (you pull it; you do not edit it) ──────────────────────────────────────────────────┐
│                                                                                                          │
│ module01/providers_utils.py   THE TOOLS (reads .env on import: load_dotenv(override=True))               │
│    groq_client · gemini_client · ollama_client     one client per provider, same OpenAI library          │
│    list_gemini_models · pick_gemini_model          ask Google which Flash models exist, pick one         │
│    ask(client, model, system, user, ...)           THE one function: one call, any provider              │
│    classify_ticket(...) + CLASSIFIER_SYSTEM        ask() specialised: ticket in, one label out           │
│                                                                                                          │
│ module01/tickets_data.py      THE DATA                                                                   │
│    CLEAR_TICKETS      3 tickets with an obvious label (billing / technical / general)                    │
│    BORDERLINE_TICKETS 3 tickets that could honestly be two labels   ── the temperature sweep             │
│                                                                                                          │
│ module01/s11_providers.ipynb  ◄── YOU ARE HERE                                                           │
│ module01/reading_providers.md, terms_s11.md    the pre-read and the terms table                          │
│ tests/test_s11.py             the checkpoint: 6 offline tests + 2 live tests (Groq, Gemini)              │
│ .env.example                  the template you copied to .env in Session 2 (Gemini block added)          │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                         │ the notebook imports the tools; the tools read ▼ your .env
┌── YOUR MACHINE  (yours; never committed) ────────────────────────────────────────────────────────────────┐
│ .env           BASE_URL · API_KEY · MODEL (Groq)   GEMINI_API_KEY · GEMINI_MODEL (Gemini)                │
│ Ollama         optional; listens on http://localhost:11434 ; model llama3.2:3b                           │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                         │ every ask() becomes one web request ▼
          Groq API (api.groq.com)      Gemini API (generativelanguage.googleapis.com)      Ollama (your laptop)
```

No work-repository file is written today. What you produce is on your screen (answers, token counts, the sweep) plus four reflection answers in section 7.

### What happens to one prompt

```
 "What is an EMI?"            "You are a terse assistant..."         client = groq / gemini / ollama
        │ user                        │ system (may be "")                   model = "openai/gpt-oss-20b" ...
        ▼                             ▼                                      │
   ask(client, model, system, user, temperature=0.0, max_tokens=200, stop=None)
        │
        │ 1. build the messages list   [ {system: ...} (only if system is not ""),  {user: ...} ]
        │ 2. build the settings        model, messages, temperature, max_tokens (+ stop, only if given)
        ▼
   client.chat.completions.create(**settings)  ──►  POST  <BASE_URL>/chat/completions  ──►  provider
                                                                                              │
        ┌──────────────────────── the reply comes back in the SAME shape from every provider ◄┘
        ▼
   { "text": r.choices[0].message.content,
     "prompt_tokens": ..., "completion_tokens": ..., "total_tokens": ... }      ◄── what you print
```

`classify_ticket()` is the same path with a fixed system message (`CLASSIFIER_SYSTEM`) and `max_tokens=1000` (a ceiling with room for a reasoning model's hidden thinking: with 10, `gpt-oss-20b` used the whole budget thinking and replied with nothing); it then cleans the reply and checks it is one of three allowed labels.

### The helpers you will call, and what each does

| Helper | Lives in | You give it | It gives back | Used in section |
|---|---|---|---|---|
| `groq_client()` | providers_utils | nothing (reads `BASE_URL`, `API_KEY`) | a client for the provider in `.env` (Groq by default) | 1 |
| `gemini_client()` | providers_utils | nothing (reads `GEMINI_API_KEY`) | a Gemini client; `RuntimeError` if the key is missing | 1 |
| `ollama_client()` | providers_utils | nothing | a client for `localhost:11434/v1` (does not check Ollama runs) | 1 |
| `list_gemini_models()` | providers_utils | nothing (asks Google live) | the chat-capable Flash model names your key can see | 1 |
| `pick_gemini_model(names)` | providers_utils | that list | `gemini-flash-latest` if offered, else the highest plain `gemini-X.Y-flash`, else `None` | 1 |
| `ask(client, model, system, user, ...)` | providers_utils | a client, a model name, two texts, optional `temperature`, `max_tokens`, `stop` | `{"text", "prompt_tokens", "completion_tokens", "total_tokens"}` | 2, 3, 4 |
| `classify_ticket(client, model, text, temperature=...)` | providers_utils | a client, a model, one ticket | everything `ask()` returns, plus `label` and `valid` | 5 |
| `CLEAR_TICKETS`, `BORDERLINE_TICKETS`, `ALL_TICKETS` | tickets_data | (data) | lists of `(ticket_id, text)` pairs | 5 |

Three **under the hood** cells rebuild `pick_gemini_model`, `ask` and the label cleaning in `classify_ticket` in front of you, and check they match the real helpers, with no model call.

### The order of the sections

| # | Section | Calls the model? | Minutes |
|---|---|---|---|
| 1 | Setup: three clients, one function | no model call (one request to list Gemini models, if you have a key) | 3 |
| 2 | The messages format, properly | yes, 2 calls (Groq) | 5 |
| 3 | Same prompt, three providers | yes, up to 3 calls (one per provider) | 7 |
| 4 | Sampling: temperature, top_p, max_tokens, stop | yes, 3 calls (Groq) | 8 |
| 5 | The temperature sweep | yes, 36 short calls (3 tickets × 4 temperatures × 3 runs) | 12 |
| 6 | Switching provider: three lines, no code change | no (you re-run section 3 afterwards) | 5 |
| 7 | Reflection | no | — |

## A5. What you will be able to do, and what you will submit

**By the end of this session you will be able to:**
1. Call Groq, Gemini and a local Ollama model through one function, changing only the client and model name.
2. Explain what the system message does, and show the difference with and without it.
3. Read the token usage of a reply and explain why the numbers differ between providers.
4. Set `max_tokens` and `stop` on purpose, and predict how a reply will be cut.
5. Run a temperature sweep, read "stable" versus "DRIFTED", and choose a temperature for a routing decision.
6. Switch the course's default provider by editing three lines in `.env`, with no code change.

**What you submit:**

| What | Where | How it is checked |
|---|---|---|
| The notebook run top to bottom, with your outputs | your copy of `module01/s11_providers.ipynb` | the instructor looks at section 5's sweep |
| Answers to Questions 1–4 | section 7 of this notebook | read in the debrief |
| The checkpoint | terminal: `uv run pytest tests/test_s11.py -rs` | 8 tests: 6 offline + 1 live Groq + 1 live Gemini |

**The checkpoint:** with both keys, `8 passed`. With no keys, `6 passed, 2 skipped`, and **a skipped test is not a pass**. `-rs` prints why each test was skipped (`no Groq key in .env`, `no Gemini key in .env`).

**Budget:** light. About 44 calls, most very short (the 36 sweep calls allow 10 reply tokens each). Section 3's `tokens:` lines show real counts. Free tiers limit requests per minute, so a `429` is possible when the whole class runs at once: wait and re-run.

---
# Part B · Hands-on

## 1. Setup — three clients, one function

### What and why
The notebook must find the helper file, read your `.env`, and create one **client** per **provider**. Some of you will have all three providers, some only Groq, some none yet. The setup cell must cope with all of these, say clearly what it found, and let the notebook continue.

The one function you use all day is `ask()`: build a **messages list**, call `chat.completions.create`, read the text and the **token usage**. Only the client and the model name change between providers.

### The idea: a provider is a place; a model is a brain
```
            same OpenAI library, three different "places"
   ┌──────────────┐       ┌──────────────────────────────┐       ┌──────────────────────┐
   │ groq client  │       │ gemini client                │       │ ollama client        │
   │ BASE_URL     │       │ generativelanguage.google... │       │ localhost:11434/v1   │
   │ API_KEY      │       │ GEMINI_API_KEY               │       │ key "ollama" (dummy) │
   └──────┬───────┘       └──────────────┬───────────────┘       └──────────┬───────────┘
          │ MODEL from .env              │ a Flash model, checked live      │ "llama3.2:3b"
          └───────────────► ask(client, model, system, user) ◄──────────────┘
```
**Analogy — the same phone, three numbers.** You do not buy a new phone to call a new bank; you dial a different number. The client is the phone; `base_url` and `api_key` are the number and your PIN.

**Why check Gemini names live?** Gemini's Flash model names change often, and old ones are retired for new accounts. So the setup asks Google which models *your* key can see and picks one: check, don't memorise.

**Getting a free Gemini key (2 minutes):** aistudio.google.com → sign in → "Get API key" → create. No card needed. Open `.env` with `code .env` (PowerShell, course folder) and add `GEMINI_API_KEY=...` with no quotes and no trailing space. Save, then **restart the kernel**.

### What the next cell does
1. **`sys.path[:0] = [...]`** puts the course folders at the *front* of Python's search list (`[:0]` is a slice meaning "before the first item"), so the import works from the repo root or from `module01/`.
2. **Importing `providers_utils` reads your `.env`**: it calls `load_dotenv(override=True)`, so `os.getenv("MODEL")` now sees your settings (*override*: `.env` wins over older values).
3. **The Groq line (changed for the no-key rule):** `groq = groq_client() if HAS_GROQ else None` is a **conditional expression**, Python's one-line if/else. `HAS_GROQ` is `True` only if `API_KEY` exists and is not the `paste_your...` placeholder (the test file's rule). With a real key, nothing changes.
4. **`try: ... except Exception as e:`** runs the Gemini block and, if anything fails (no key, no internet, bad key), runs the `except` part instead of crashing; `e` holds the error. `gemini, GEMINI_MODEL = None, None` sets two variables at once.
5. **`os.getenv("GEMINI_MODEL") or pick_gemini_model(...)`**: `A or B` gives `A` if it has a value, else `B`. A pinned name wins; a blank `GEMINI_MODEL=` falls through to the live pick.
6. **The Ollama block** never fails, even with Ollama stopped, because creating a client sends nothing. Section 3 sends the first real request.

In [1]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from providers_utils import ask, classify_ticket, gemini_client, groq_client, list_gemini_models, ollama_client, pick_gemini_model
from tickets_data import ALL_TICKETS, BORDERLINE_TICKETS, CLEAR_TICKETS

# providers_utils has already read .env (load_dotenv(override=True)). No real Groq key -> groq stays None,
# and every Groq cell below prints "skipped" instead of crashing. With a real key nothing changes.
HAS_GROQ = bool(os.getenv("API_KEY")) and "paste_your" not in os.getenv("API_KEY", "")
groq = groq_client() if HAS_GROQ else None
GROQ_MODEL = os.getenv("MODEL")
if groq is None:
    print("Groq not configured yet: no real API_KEY in .env (Groq cells will print 'skipped')")

try:
    gemini = gemini_client()
    # Gemini's Flash model name changes often - verify instead of trusting a name written down months ago.
    flash_models = list_gemini_models()
    GEMINI_MODEL = os.getenv("GEMINI_MODEL") or pick_gemini_model(flash_models)
    print("Gemini models available:", flash_models)
    print("using:", GEMINI_MODEL)
except Exception as e:
    gemini, GEMINI_MODEL = None, None
    print("Gemini not configured yet:", e)

try:
    ollama = ollama_client()
    OLLAMA_MODEL = "llama3.2:3b"
except Exception as e:
    ollama = None
    print("Ollama not available:", e)

Gemini models available: ['gemini-2.5-flash', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite', 'gemini-3.1-flash-lite-preview', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-flash-latest', 'gemini-flash-lite-latest']
using: gemini-3.8-flash


### What you should see
With both keys, two lines like these (*an illustration, not a real run*; your list will be whatever Google offers your key today):
```
Gemini models available: ['gemini-2.5-flash', 'gemini-2.5-flash-lite', 'gemini-flash-latest']
using: gemini-flash-latest
```
With no keys at all, you see instead:
```
Groq not configured yet: no real API_KEY in .env (Groq cells will print 'skipped')
Gemini not configured yet: GEMINI_API_KEY is not set in .env — see the .env.example block.
```
Nothing is printed for Ollama, even if it is not installed (step 6).

| If you see... | It means... | Do this |
|---|---|---|
| `Groq not configured yet` but you have a key | `.env` missing from the course folder, or still the placeholder | fix `.env` (not `.env.example`), save, **restart the kernel** |
| `Gemini not configured yet:` + a connection or `401` error | no internet, a proxy, or a mistyped key | check Wi-Fi; copy the key again |
| `using: None` | no plain Flash name in your list | pin a printed name as `GEMINI_MODEL=...`, restart |
| `ModuleNotFoundError: providers_utils` | repo out of date, or notebook opened elsewhere | `git pull`; open the notebook from the course folder |

### Under the hood: how the Gemini model is picked
`list_gemini_models()` keeps only the chat-capable Flash names; `pick_gemini_model()` chooses one. You rebuild both and compare with the real helpers on a **practice list** (like the one in the test file; made up, not a live answer from Google). No key, no internet.

**What the next cell does:**
1. `PRACTICE_IDS`: model ids in the shape Google returns them, with a `models/` prefix.
2. `my_flash_chat_models(ids)` drops the prefix (`str.removeprefix`) and keeps names that contain `flash` and none of the words in `_NON_CHAT` (image, tts, audio, live, embed, omni, native). `[n for n in names if condition]` is a **list comprehension**: "every `n` in `names` that passes the condition". `any(...)` is `True` if at least one bad word appears.
3. `my_pick(available)` returns `gemini-flash-latest` if listed. Otherwise `re.fullmatch` (a pattern that must match the *whole* name) finds plain names like `gemini-2.5-flash`, and `max()` on `(version, name)` pairs returns the highest version.
4. To test the real `list_gemini_models()` offline, the cell swaps in a fake `gemini_client` for a moment; `finally:` (which always runs) puts the real one back.
5. It prints both results and `same` or `DIFFERENT`.

In [2]:
import re
import providers_utils
from types import SimpleNamespace
from providers_utils import _NON_CHAT

PRACTICE_IDS = ["models/gemini-2.5-flash", "models/gemini-2.5-flash-image", "models/gemini-2.5-flash-lite",
                "models/gemini-2.5-flash-preview-tts", "models/gemini-3-flash-preview", "models/gemini-3.5-flash",
                "models/gemini-flash-latest", "models/gemini-omni-1.1-flash"]   # practice names, not a live list

def my_flash_chat_models(ids):
    names = [i.removeprefix("models/") for i in ids]                              # drop the "models/" prefix
    return sorted([n for n in names if "flash" in n.lower() and not any(bad in n.lower() for bad in _NON_CHAT)])

def my_pick(available):
    if "gemini-flash-latest" in available:                                        # the alias always points at a current model
        return "gemini-flash-latest"
    versions = []
    for n in available:
        m = re.fullmatch(r"gemini-(\d+(?:\.\d+)?)-flash", n)                      # plain names only: no -lite, no -preview
        if m:
            versions.append((float(m.group(1)), n))
    return max(versions)[1] if versions else None

fake = SimpleNamespace(models=SimpleNamespace(list=lambda: [SimpleNamespace(id=i) for i in PRACTICE_IDS]))
real_client = providers_utils.gemini_client
try:
    providers_utils.gemini_client = lambda: fake                                  # offline: the helper gets our fake list
    real_names = providers_utils.list_gemini_models()
finally:
    providers_utils.gemini_client = real_client                                   # always put the real one back

mine_names = my_flash_chat_models(PRACTICE_IDS)
print("mine:", mine_names)
print("real:", real_names, "->", "same" if mine_names == real_names else "DIFFERENT")
without_alias = [n for n in mine_names if n != "gemini-flash-latest"]
for names in (mine_names, without_alias):
    print(f"pick from {len(names)} names: mine={my_pick(names)!r} real={pick_gemini_model(names)!r}",
          "same" if my_pick(names) == pick_gemini_model(names) else "DIFFERENT")

mine: ['gemini-2.5-flash', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview', 'gemini-3.5-flash', 'gemini-flash-latest']
real: ['gemini-2.5-flash', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview', 'gemini-3.5-flash', 'gemini-flash-latest'] -> same
pick from 5 names: mine='gemini-flash-latest' real='gemini-flash-latest' same
pick from 4 names: mine='gemini-3.5-flash' real='gemini-3.5-flash' same


**What you should see:** both name lists equal (`same`). The image, tts and omni names are gone. Look closely at two that *stay*: `gemini-2.5-flash-lite` and `gemini-3-flash-preview` are chat models, so the filter keeps them, but `my_pick` never chooses them because they are not plain `gemini-X.Y-flash` names. Then two pick lines, both `same`: with the alias present the pick is `'gemini-flash-latest'`; without it, the highest plain version, `'gemini-3.5-flash'`.

**`SimpleNamespace`** makes a quick object with any attributes you give it (here, a fake client); **`lambda`** is a one-line function without `def`. **The take-away:** "verify the model name" is a filter and a rule, readable in fifteen lines.

### Check yourself
*Your `.env` has `GEMINI_MODEL=gemini-2.5-flash` pinned, but the setup cell lists `gemini-flash-latest` too. Which one is used, and why?* (Hint: step 5, `A or B`.)

> **Common mistake:** editing `.env` and re-running only this cell. The clients already exist with the old values. Restart the kernel after any change to `.env`, then run from the top.

## 2. The messages format, properly

### What and why
You have used this shape since `hello.py`: a list of role/content pairs. Later sessions (S19 chat memory, S20 tools, Module 6 agents) add more messages to this same list. Two things worth seeing deliberately:

- The **system** message sets who the model is. Drop it and the model falls back to a generic default persona.
- **Order and role matter**: a system instruction buried as a second user message is weaker than a real system message.

### The idea
```
 WITH a system message                               WITHOUT one (system = "")
 ──────────────────────────────────────────          ──────────────────────────────────────────
 [ {"role": "system",                                [ {"role": "user",
    "content": "You are a terse assistant               "content": "What is an EMI?"} ]
     for a retail bank. Answer in one short
     sentence."},                                     ▲ ask() leaves the system message OUT
   {"role": "user",                                     completely when system is "" (it does
    "content": "What is an EMI?"} ]                     not send an empty one)
```
**Analogy — the job description.** A new clerk with a written job description ("retail bank; one sentence per answer") behaves predictably. Without one, they answer in whatever style they are used to. The system message is that job description, sent with *every* request.

**Predict first:** without the system message, will the answer be shorter or longer? Will it still mention banking?

### What the next cell does
1. Stores the system text and the question in `system` and `user`.
2. **Changed for the no-key rule:** `if groq is None:` prints `skipped (...)`; under `else:` is the original code, just indented.
3. Calls `ask()` twice: with the system text, and with `""` (no system message).
4. `ask()` returns a **dictionary**; `with_system["text"]` reads the value under the key `"text"`. `"\n "` in a string is a line break plus a space.

In [3]:
system = "You are a terse assistant for a retail bank. Answer in one short sentence."
user = "What is an EMI?"

if groq is None:
    print("skipped (no Groq key in .env)")
else:
    with_system = ask(groq, GROQ_MODEL, system, user)
    without_system = ask(groq, GROQ_MODEL, "", user)

    print("WITH system message:\n ", with_system["text"])
    print("\nWITHOUT a system message:\n ", without_system["text"])

WITH system message:
  EMI is the fixed monthly payment you make to repay a loan.

WITHOUT a system message:
  


### What you should see
*An illustration, not a real run* (your wording will differ):
```
WITH system message:
  An EMI is a fixed monthly payment you make to repay a loan, covering both interest and principal.

WITHOUT a system message:
  An EMI (Equated Monthly Instalment) is a fixed payment amount made by a borrower to a lender at a
  specified date each month. EMIs are applied to both interest and principal... (several more sentences)
```
**How to read it:** the first answer follows the job description; the second is usually longer and more general. If both look alike, that is a finding too: some models are terse by default. `AuthenticationError ... 401` means a wrong or revoked key (Part C).

### Under the hood: build `ask()` yourself
`ask()` is about fifteen lines. You write it yourself and prove it sends *exactly* the same request as the real one, using a **fake client** that records the request instead of calling the internet.

**What the next cell does:**
1. `FakeClient` is a small **class**. Its `create(**kwargs)` stores what it received in `self.sent` and returns a made-up reply shaped like a real one. In a definition, `**kwargs` collects keyword arguments into a dictionary; in a call, `**kwargs` spreads a dictionary back out into keyword arguments.
2. `my_ask(...)` builds the messages (system only `if system:`), a `dict(...)` of settings, adds `stop` only if given, and calls `create(**kwargs)`. The bare `*` in the signature means later arguments must be named (`max_tokens=15`).
3. For three cases it runs the real `ask()` on one fake and `my_ask()` on another, and compares requests and results.
4. It prints the messages sent "without system".

In [4]:
class FakeClient:
    """Looks like an OpenAI client to ask(), but never touches the network: it records the request."""
    def __init__(self, reply="An EMI is a fixed monthly loan repayment."):
        self.reply, self.sent = reply, None
        self.chat = SimpleNamespace(completions=self)              # so client.chat.completions.create(...) works
    def create(self, **kwargs):
        self.sent = kwargs                                         # remember exactly what would be sent
        usage = SimpleNamespace(prompt_tokens=21, completion_tokens=9, total_tokens=30)   # made-up numbers
        return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=self.reply))], usage=usage)

def my_ask(client, model, system, user, *, temperature=0.0, max_tokens=200, stop=None):
    messages = []
    if system:                                                     # "" means: no system message at all
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user})
    kwargs = dict(model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    if stop:
        kwargs["stop"] = stop                                      # only sent when you ask for it
    r = client.chat.completions.create(**kwargs)
    return {"text": r.choices[0].message.content, "prompt_tokens": r.usage.prompt_tokens,
            "completion_tokens": r.usage.completion_tokens, "total_tokens": r.usage.total_tokens}

cases = [("with system", dict(system=system, user=user)),
         ("without system", dict(system="", user=user)),
         ("with stop", dict(system="Answer briefly.", user=user, max_tokens=50, stop=["\n"]))]
for name, args in cases:
    real_fake, my_fake = FakeClient(), FakeClient()
    real_out = ask(real_fake, "any-model", **args)
    mine_out = my_ask(my_fake, "any-model", **args)
    same = real_fake.sent == my_fake.sent and real_out == mine_out
    print(f"{name:<15} request same: {real_fake.sent == my_fake.sent}   result same: {real_out == mine_out}   ->",
          "same" if same else "DIFFERENT")
    if name == "without system":
        print("   messages sent:", my_fake.sent["messages"])

with system     request same: True   result same: True   -> same
without system  request same: True   result same: True   -> same
   messages sent: [{'role': 'user', 'content': 'What is an EMI?'}]
with stop       request same: True   result same: True   -> same


**What you should see:** three lines ending in `same`, and a messages list with only the user message. Change one detail in `my_ask` and re-run: `DIFFERENT`. That is how you know the check is real. **The take-away:** a model call is a dictionary of settings sent to an address; every provider receives the same dictionary.

### Check yourself
*Question 1 — What changed between the two answers, and which field in `.env` would you edit to make this run against Gemini instead of Groq?*

> **Common mistake:** putting standing rules in the user message because "it works" in a quick test. The system role is where models expect rules, and keeping them there lets the user text change freely.

## 3. Same prompt, three providers

### What and why
Same system prompt, same question, three providers. Compare the text and the token usage. This is the entire point of today: the code around this cell does not change — only which client and model name go in.

If your code only ran against one vendor, you could not tell whether it is truly **provider-agnostic**. One loop over three providers proves it.

### The idea
```
   prompt + system ──┬──► ask(groq,   "openai/gpt-oss-20b" (from .env)) ──► text + tokens
                     ├──► ask(gemini, GEMINI_MODEL (picked live))       ──► text + tokens
                     └──► ask(ollama, "llama3.2:3b")                    ──► text + tokens
                              ▲
                              └── the SAME function; only the first two arguments change
```
**Analogy — same question, three consultants.** Their answers differ in style and length, and each bills in their own units. The comparison is fair only because the question is identical.

**Token counts are not directly comparable:** each provider has its own tokenizer (Session 9), and a reasoning model's completion count can include hidden thinking tokens. Fewer tokens is not automatically cheaper: prices per token differ (Session 10).

### What the next cell does
1. `results = {}` is an empty dictionary to collect each reply under its label.
2. The `for` loop walks a list of **tuples** `(label, client, model)` and unpacks each into three variables.
3. `if client is None or model is None:` prints `skipped (not configured)`; `continue` jumps to the next provider. That is why this cell needed **no change**.
4. `try:` sends the request. If it fails (Ollama stopped, rate limit, retired model), `except` prints the error type (`type(e).__name__`) and the first 100 characters (`str(e)[:100]`), and the loop carries on.
5. **f-strings** such as `f"--- {label} ({model})"` put variables' values inside the text.

In [5]:
prompt = "In one sentence, what is the difference between a workflow and an agent?"
results = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("gemini", gemini, GEMINI_MODEL), ("ollama", ollama, "llama3.2:3b" if ollama else None)]:
    if client is None or model is None:
        print(f"--- {label}: skipped (not configured)\n"); continue
    try:
        r = ask(client, model, "You are a concise AI engineering tutor.", prompt)
        results[label] = r
        print(f"--- {label} ({model})")
        print(" ", r["text"].strip())
        print(f"   tokens: prompt={r['prompt_tokens']} completion={r['completion_tokens']} total={r['total_tokens']}\n")
    except Exception as e:
        print(f"--- {label}: error ({type(e).__name__}: {str(e)[:100]})\n")

--- groq (openai/gpt-oss-20b)
  A workflow is a predefined, orchestrated sequence of tasks, while an agent is an autonomous entity that perceives, decides, and acts to achieve goals.
   tokens: prompt=97 completion=133 total=230



--- gemini: error (InternalServerError: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high dem)



--- ollama (llama3.2:3b)
  A workflow is a series of tasks or processes that are executed in a specific order, whereas an agent is a software entity that autonomously interacts with the environment, making decisions and taking actions to achieve a goal.
   tokens: prompt=48 completion=43 total=91



### What you should see
*An illustration, not a real run* (all three working; numbers made up):
```
--- groq (openai/gpt-oss-20b)
  A workflow follows a fixed sequence of steps defined in code, while an agent lets the model decide
  which steps to take next.
   tokens: prompt=85 completion=60 total=145

--- gemini (gemini-flash-latest)
  A workflow executes predefined steps, whereas an agent dynamically decides its own actions.
   tokens: prompt=24 completion=17 total=41

--- ollama: error (APIConnectionError: Connection error.)
```
**How to read it:** similar meaning, different words: different models, different training. Compare correctness, not wording. Note which `tokens:` line is smallest (Question 2).

With no keys and no Ollama: `groq: skipped`, `gemini: skipped`, and `ollama: error (APIConnectionError ...)`, because nothing listens on your laptop. The notebook continues.

| If you see... | It means... | Do this |
|---|---|---|
| `ollama: error (APIConnectionError ...)` | Ollama not running | optional; start the Ollama app or run `ollama serve` in a second terminal |
| `ollama: error (NotFoundError ...)` | model not downloaded | `ollama pull llama3.2:3b` |
| `RateLimitError` | `429`, too many requests per minute | wait 30–60 seconds |

### Check yourself
*Question 2 — Which provider used the fewest total tokens for the same question? Does that necessarily mean it is the cheapest or the best choice — what else would you need to know (hint: Session 10)?*

> **Common mistake:** calling one provider "better" after one question. To choose, compare many prompts on quality, price, speed and data rules (the Session 10 matrix).

## 4. Sampling: temperature, top_p, max_tokens, stop

### What and why
A model picks **one token at a time** (Session 9). At each step it *samples* one token from a list of possible next tokens, and four *sampling* settings shape that pick.

- **temperature** (0 to roughly 2) controls how much randomness goes into picking the next token. 0 is as close to deterministic as a provider allows; higher values let the model pick less-likely tokens more often.
- **top_p** is an alternative way to control the same thing — restrict the choice to the smallest set of tokens whose probabilities add up to `top_p`. Most labs here use temperature alone; the two are rarely combined.
- **max_tokens** caps the length of the reply. Set it too low on a reasoning model and you can get an empty answer, as you saw in Session 10.
- **stop** ends generation the moment a given string appears — useful for forcing a short, structured answer.

### The idea: what temperature does to the choice
At each step every possible next token gets a score; temperature reshapes the scores before the pick. The numbers below are **made up for illustration**, but the effect is real:

```
 chance of each label at one step         temperature 0.3      temperature 0.7      temperature 1.0
 ───────────────────────────────          ───────────────      ───────────────      ───────────────
 billing                                  ████████████  79%    █████████    59%     ████████     53%
 technical                                ███           21%    █████        34%     █████        35%
 general                                  ▏              1%    █             7%     ██           12%
                                          ▲ sharp: nearly always billing           ▲ flat: often not
```
At temperature 0 the top choice is (nearly) always taken; as temperature rises, the chances flatten. **top_p** works differently: with `top_p=0.8` and the 1.0 column, only billing and technical stay on the shortlist (53% + 35% ≥ 80%); general is never picked.

**Analogy.** Temperature is how loosely a manager follows their first instinct; top_p is a shortlist; max_tokens is a page limit that stops the writer mid-sentence; stop is a tripwire that takes the pen away. `ask()` exposes temperature, max_tokens and stop, but **not** top_p, on purpose: one dial at a time.

### What the next cell does (max_tokens)
1. Calls `ask()` twice with the same question; the only difference is the **keyword argument** `max_tokens=15` versus `150`.
2. Prints the short reply, and the first 300 characters of the long one (`[:300]`).
3. **Changed for the no-key rule:** wrapped in `if groq is None: ... else:`.

**Predict first:** will the 15-token reply be one complete short sentence, or will it stop in the middle?

In [6]:
if groq is None:
    print("skipped (no Groq key in .env)")
else:
    short = ask(groq, GROQ_MODEL, "You are a helpful assistant.", "Explain what a vector database is.", max_tokens=15)
    enough = ask(groq, GROQ_MODEL, "You are a helpful assistant.", "Explain what a vector database is.", max_tokens=150)
    print("max_tokens=15:\n ", short["text"], "\n")
    print("max_tokens=150:\n ", enough["text"][:300])

max_tokens=15:
   

max_tokens=150:
  


### What you should see (max_tokens)
*An illustration, not a real run:*
```
max_tokens=15:
  A vector database is a specialized database designed to store and

max_tokens=150:
  A vector database is a specialized database designed to store, index, and search high-dimensional
  vectors (embeddings). Instead of matching exact values, it finds items that are *similar*...
```
**How to read it:** the 15-token reply stops mid-sentence: the model did not plan a short answer, it was cut off. With 150 the answer completes.

The course model, `openai/gpt-oss-20b`, is a **reasoning model**: its hidden thinking counts against `max_tokens` too, so the 15-token reply may be **empty** (`None` or blank). That is the Session 10 trap in your own output. If even the 150 line fails with `TypeError: 'NoneType' ...`, the whole budget went on thinking; tell the instructor rather than changing the numbers.

### What the next cell does (stop)
1. Asks for "three benefits of retrieval-augmented generation" with `max_tokens=200` and `stop=["\n"]`: a list with one stop string, the line break. (A call may span two lines inside brackets.)
2. Prints the reply. **Changed for the no-key rule** in the same way.

**Predict first:** a list of three benefits is usually written on three lines. What will you get if generation stops at the first line break?

In [7]:
if groq is None:
    print("skipped (no Groq key in .env)")
else:
    r = ask(groq, GROQ_MODEL, "You are a helpful assistant. Answer in a short paragraph.",
            "List three benefits of retrieval-augmented generation.", max_tokens=200, stop=["\n"])
    print(r["text"])

Retrieval‑augmented generation (RAG) boosts a model’s usefulness by (1) grounding responses in real documents, which cuts hallucinations and improves factual accuracy; (2) allowing the system to pull in up‑to‑date or domain‑specific information that the base model may not know, keeping answers current and relevant; and (3) reducing the need for massive pre‑training, since the retrieval component supplies the bulk of the knowledge, making the approach more efficient and easier to adapt to new topics.


### What you should see (stop)
*An illustration, not a real run:* one line only, for example
```
RAG grounds answers in your own documents, keeps knowledge current without retraining, and makes answers traceable.
```
If the model wrote a list, you get only the first item or a heading like `Here are three benefits:`. The `\n` itself is never included. An empty reply means the first output was a line break. Stop sequences are a cheap way to force a shape and save tokens; Session 13 adds a stronger tool, structured (JSON) output.

### Check yourself
*You need a one-word label from a model. Which two of today's four settings would you set, and to what values?* (You will see the course's answer in section 5: look at `classify_ticket`.)

> **Common mistake:** treating `max_tokens` as "the length I want". It is a ceiling that cuts. *Ask* for a short answer in the system message and keep `max_tokens` as a safety limit with room to spare.

## 5. The temperature sweep — where it actually matters

### What and why
Three clear tickets, three borderline ones. Run the same ticket through the classifier at several temperatures and watch what happens. A routing decision like this should usually be made at temperature 0 — you do not want "billing" vs "technical" to depend on a coin flip.

This is the heart of today: you produce the A1 story's Monday/Tuesday effect on purpose, and find the setting that stops it.

### The idea
The classifier is `classify_ticket()`: `ask()` with a fixed system message, `CLASSIFIER_SYSTEM` ("classify ... into exactly one category: billing, technical, or general. Reply with only that one word"), and `max_tokens=1000`. Why so high for a one-word answer? The course model thinks before it answers, and the hidden thinking is paid from the same budget: an early version used 10 and got empty replies.

```
 one BORDERLINE ticket
        │
        ├── temperature 0.0 ──► run 1, run 2, run 3 ──► ['billing', 'billing', 'billing']      stable
        ├── temperature 0.3 ──► run 1, run 2, run 3 ──► ['billing', 'billing', 'billing']      stable
        ├── temperature 0.7 ──► run 1, run 2, run 3 ──► ['billing', 'technical', 'billing']    DRIFTED
        └── temperature 1.0 ──► run 1, run 2, run 3 ──► ['technical', 'billing', 'technical']  DRIFTED
                                       (an illustration of the pattern, not a real run)
```
**Why borderline tickets?** A clear ticket ("charged twice... refund the duplicate") gives *billing* such a high chance that even a flat distribution picks it. A borderline one ("still charged the old price **and** the new features aren't showing") has two labels with similar chances; that is where temperature changes the answer.

**Analogy — a coin flip hidden inside a decision.** Nobody sees the coin; they only see a billing problem that sat in the technical queue for three days.

**Three runs is a small sample.** "stable" does not prove a temperature is safe; one "DRIFTED" proves the label *can* change.

### What the next cell does
1. `TEMPERATURES` and `RUNS_PER_TEMPERATURE` are in capitals, the Python convention for settings you may change.
2. `if groq is None:` was already in the original, so this cell is **unchanged**.
3. Three nested loops: each borderline ticket, each temperature, three runs. `for _ in range(...)`: `_` names a counter you do not need.
4. A conditional expression stores the label if valid, else `INVALID:` plus the raw reply; `!r` in the f-string shows it with quotes, so an empty reply appears as `''`.
5. `len(set(labels)) == 1`: a `set` keeps distinct values only, so one element means `stable`.
6. `{temp:<4}` and `{...:8}` pad values so the columns line up.

That is 36 short calls; it takes a little while. Do not click Stop.

In [8]:
TEMPERATURES = [0.0, 0.3, 0.7, 1.0]
RUNS_PER_TEMPERATURE = 3

if groq is None:
    print("skipped (no Groq client)")
else:
    for tid, text in BORDERLINE_TICKETS:
        print(f"\n=== {tid} ===\n{text}\n")
        for temp in TEMPERATURES:
            labels = []
            for _ in range(RUNS_PER_TEMPERATURE):
                result = classify_ticket(groq, GROQ_MODEL, text, temperature=temp)
                labels.append(result["label"] if result["valid"] else f"INVALID:{result['label']!r}")
            stable = len(set(labels)) == 1
            print(f"  temperature={temp:<4} {'stable' if stable else 'DRIFTED':8} {labels}")



=== borderline_1 ===
I upgraded my plan but I'm still being charged the old price and the new features aren't showing up either.



  temperature=0.0  stable   ['billing', 'billing', 'billing']


  temperature=0.3  stable   ['billing', 'billing', 'billing']


  temperature=0.7  stable   ['billing', 'billing', 'billing']


  temperature=1.0  stable   ['billing', 'billing', 'billing']

=== borderline_2 ===
Your app keeps logging me out and now I can't tell if I'm still being billed for the plan I signed up for.



  temperature=0.0  stable   ['billing', 'billing', 'billing']


... (3 outputs left out)

  temperature=0.0  stable   ['technical', 'technical', 'technical']


  temperature=0.3  stable   ['technical', 'technical', 'technical']


  temperature=0.7  DRIFTED  ['technical', 'technical', 'general']


  temperature=1.0  stable   ['technical', 'technical', 'technical']


### What you should see
For each borderline ticket, its id and text, then four lines. *An illustration, not a real run:*
```
=== borderline_1 ===
I upgraded my plan but I'm still being charged the old price and the new features aren't showing up either.

  temperature=0.0  stable   ['billing', 'billing', 'billing']
  temperature=0.3  stable   ['billing', 'billing', 'billing']
  temperature=0.7  DRIFTED  ['billing', 'technical', 'billing']
  temperature=1.0  DRIFTED  ['technical', 'billing', 'billing']
```
**How to read it:** go down the temperatures and find the first `DRIFTED`. A common pattern is no change at 0, an occasional flip between 0.3 and 0.7, more at 1.0. **Your result may differ**: report what *you* saw.

| If you see... | It means... | Do this |
|---|---|---|
| `stable` everywhere | a confident model, or three runs missed a change | a real finding; try `RUNS_PER_TEMPERATURE = 5` if you have quota |
| `DRIFTED` at 0.0 | temperature 0 is close to deterministic, not guaranteed | note it; it is why classifiers are also measured (S17) |
| `INVALID:'billing or technical'` | the model ignored "one word only" | the check did its job; count it as a failure |
| `INVALID:''` | empty reply: the 10-token budget went on hidden thinking | possible on a reasoning model; mention it in Question 3 |

### Under the hood: how `classify_ticket` cleans the reply
Replies are rarely exactly `billing`; they may be `Billing.` or ` TECHNICAL\n`. You rebuild the cleaning rule and compare it with the real helper, using section 2's `FakeClient`, so no model is called.

**What the next cell does:**
1. `my_read_label(text)` chains `.strip()` (spaces and line breaks off both ends), `.lower()`, and `.strip(".")` (full stops off both ends), then checks membership in `{"billing", "technical", "general"}`.
2. For six replies it compares the real `classify_ticket()` (on a fake) with yours.
3. It prints the settings `classify_ticket` sent. (The real helper returns `{**result, "label": ..., "valid": ...}`: `**result` copies `ask()`'s keys, then two are added.)

In [9]:
from providers_utils import CLASSIFIER_SYSTEM

def my_read_label(text):
    label = text.strip().lower().strip(".")                       # " Billing.\n" -> "billing"
    return label, label in {"billing", "technical", "general"}

replies = ["Billing.\n", "  TECHNICAL", "general", "Billing or technical", "billing!", ""]
for reply in replies:
    fake = FakeClient(reply)
    real = classify_ticket(fake, "any-model", "some ticket text", temperature=0.7)
    mine = my_read_label(reply)
    print(f"{reply!r:<24} real=({real['label']!r}, {real['valid']})  mine={mine}  ->",
          "same" if (real["label"], real["valid"]) == mine else "DIFFERENT")

print("\nwhat classify_ticket sent:", {k: v for k, v in fake.sent.items() if k != "messages"})
print("system message:", fake.sent["messages"][0]["content"] == CLASSIFIER_SYSTEM)

'Billing.\n'             real=('billing', True)  mine=('billing', True)  -> same
'  TECHNICAL'            real=('technical', True)  mine=('technical', True)  -> same
'general'                real=('general', True)  mine=('general', True)  -> same
'Billing or technical'   real=('billing or technical', False)  mine=('billing or technical', False)  -> same
'billing!'               real=('billing!', False)  mine=('billing!', False)  -> same
''                       real=('', False)  mine=('', False)  -> same

what classify_ticket sent: {'model': 'any-model', 'temperature': 0.7, 'max_tokens': 1000}
system message: True


**What you should see:** six lines ending in `same`. `'billing!'` is **not** valid (only full stops are removed), nor are `'Billing or technical'` and `''`. Then `temperature: 0.7`, `max_tokens: 10`, `system message: True`. `{k: v for k, v in ... if k != "messages"}` is a **dictionary comprehension**.

**The take-away:** "valid" means one of the allowed words, not the *right* word. Measuring rightness is Session 17.

### Check yourself
*Question 3 — At which temperature, if any, did the label start to change between runs on the same ticket? What would you set temperature to for a routing decision in production, and why?*
*Question 4 — Run the same sweep on one of the CLEAR_TICKETS. Does it drift as much as the borderline ones? What does that tell you about when temperature actually matters?*

To do Question 4, copy the sweep cell and change `BORDERLINE_TICKETS` to `CLEAR_TICKETS` in the copy. That is another 36 calls.

> **Common mistake:** "temperature 0 means the model is right." It makes the answer *repeatable*, not *correct*: a classifier can be consistently wrong at 0.

## 6. Switching provider — three lines, no code change

### What and why
This is the payoff. Open `.env`, comment the three Groq lines, and point `BASE_URL`/`API_KEY`/`MODEL` at a different provider. Nothing in `providers_utils.py` or this notebook changes. Re-run section 3 afterwards and compare. `groq_client()` reads whatever `.env` says, so those three lines decide which provider the whole course talks to.

### The idea
```
 .env today (Groq)                                   .env after switching (Ollama: the block already exists)
 ────────────────────────────────────────            ────────────────────────────────────────────────
 BASE_URL=https://api.groq.com/openai/v1             # BASE_URL=https://api.groq.com/openai/v1
 API_KEY=gsk_...                                     # API_KEY=gsk_...
 MODEL=openai/gpt-oss-20b                            # MODEL=openai/gpt-oss-20b
                                                     BASE_URL=http://localhost:11434/v1
 # BASE_URL=http://localhost:11434/v1                API_KEY=ollama
 # API_KEY=ollama                                    MODEL=llama3.2:3b
 # MODEL=llama3.2:3b
```
A `#` turns a line into a comment. The Ollama block is already in `.env.example`. For Gemini, write three lines yourself: `BASE_URL=https://generativelanguage.googleapis.com/v1beta/openai/`, `API_KEY=` your Gemini key, `MODEL=` the name section 1 printed after `using:`.

**Analogy — swap the plug, not the appliance.** Abroad, you change the adapter, not the laptop. `.env` is the adapter.

### The steps, in order
1. `code .env` in PowerShell (course folder).
2. `#` before the three Groq lines; remove `#` from the three Ollama lines (or write the Gemini lines). Save.
3. **Restart the kernel**: section 1's clients still hold the old address and key.
4. Run section 1's setup cell, this cell, then section 3.
5. **Switch back** to Groq afterwards and restart. The instructor pins `MODEL`.

### What the next cell does
1. `importlib.reload(providers_utils)` runs the module again, and with it `load_dotenv(override=True)`, so `os.getenv` sees the new `.env` values.
2. Re-imports `ask` and prints `BASE_URL` and `MODEL` so you can confirm the switch.

A reload refreshes *settings*, not the clients already made: hence restart and re-run section 1. This cell calls no model and is unchanged.

In [10]:
# After editing .env, restart the kernel (Kernel > Restart), then re-run section 1's setup cell
# and this cell to see the SAME code now talking to a different provider.
import importlib, providers_utils
importlib.reload(providers_utils)
from providers_utils import ask
print("BASE_URL now:", os.getenv("BASE_URL"))
print("MODEL now:", os.getenv("MODEL"))

BASE_URL now: https://api.groq.com/openai/v1
MODEL now: openai/gpt-oss-20b


### What you should see
*An illustration, not a real run*, after switching to the Ollama block:
```
BASE_URL now: http://localhost:11434/v1
MODEL now: llama3.2:3b
```
Re-run section 3: the `groq` line now shows `llama3.2:3b`, because "the client from `.env`" is now your laptop. Same code, different plug. With no `.env`, both values print `None` (`copy .env.example .env` in PowerShell; Mac/Linux `cp`).

| If you see... | It means... | Do this |
|---|---|---|
| the old values | `.env` not saved, or `.env.example` edited by mistake | save `.env`, re-run |
| `404 model not found` after switching | that `MODEL` does not exist on the new provider | Ollama: `llama3.2:3b`; Gemini: the name from section 1 |

### Check yourself
*You switched to Ollama, restarted, and re-ran section 3. Which lines of Python did you change? Which three lines of text did you change, and in which file?*

> **Common mistake:** forgetting to switch back. The course and the live tests expect the Groq lines.

## 7. Reflection

### What and why
Use what *your* notebook printed, not the illustrations.

Answer questions 1–4 above, one or two lines each.

1.
2.
3.
4.

Then run the checkpoint from the terminal: `uv run pytest tests/test_s11.py`

Add `-rs` to see skip reasons. With both keys: `8 passed`. A skip is not a pass.

> **Common mistake:** leaving `GEMINI_MODEL=` blank. The live Gemini test reads that value as it is, so it may send an empty model name and fail. Pin the name section 1 printed after `using:` before the checkpoint.

---
# Part C · Wrap-up

## C1. Summary
- A **provider** is where a model runs; a **model** is which brain answers. Your code needs only three settings to talk to any OpenAI-compatible provider: `BASE_URL`, `API_KEY`, `MODEL`, all in `.env`.
- `ask()` is one function for every provider: build a **messages list**, call `chat.completions.create`, read the text and the token usage. Only the client and the model name change.
- The **system message** is the job description. Leave it out and the model uses its default style.
- **Token counts** differ between providers for the same text (different tokenizers, hidden reasoning), and fewer tokens does not mean cheaper.
- **max_tokens** is a ceiling that cuts, and on a reasoning model the hidden thinking counts too. **stop** is a tripwire that ends the reply at a given string.
- **temperature** controls randomness. Decisions such as routing belong at 0 (or close to it): repeatable, though not guaranteed correct. Borderline inputs are where drift shows.
- Model names change: verify them live (`list_gemini_models`).

## C2. The words again

| Term | One-line meaning |
|---|---|
| Provider | Where a model runs (Groq, Google, your laptop) |
| Model | Which trained network answers, chosen by name |
| OpenAI-compatible endpoint | Another vendor's address that accepts the same request shape |
| `BASE_URL`, `API_KEY`, `MODEL` | Where, who, and which model: the three settings in `.env` |
| Client | The Python object that holds one address and one key |
| Provider-agnostic code | Code that works unchanged with any provider; only config changes |
| Messages list and roles | Role/content pairs: system (job description), user, assistant |
| Token usage | Prompt, completion and total tokens: the itemised bill |
| temperature | The randomness dial for each next token |
| top_p | A shortlist by probability: another dial for the same randomness |
| max_tokens | The ceiling on reply length; it cuts, it does not plan |
| Stop sequence | A string that ends the reply the moment it appears |
| Drift | Same input, different output across runs |

## C3. Interview practice

**1. "How would you design an LLM application so you can switch providers without rewriting code?"**
*Model answer:* Keep the provider in configuration (base URL, key, model name from the environment) and route every call through one function. Many providers accept the OpenAI-compatible shape, so one client library covers them. *In this lab:* `ask()` ran unchanged against Groq, Gemini and Ollama; switching the course default took three lines in `.env` and a kernel restart. *Honest limit:* compatibility covers common features only; model names, limits and quality still differ, so each provider must be tested.

**2. "What does temperature do, and what would you set it to for a classifier?"**
*Model answer:* Temperature reshapes the probabilities of the next token: low values make the top choice dominant, high values flatten them. For a classifier, use 0 or close to it. *In this lab:* the borderline sweep at 0, 0.3, 0.7 and 1.0 showed, through `stable` versus `DRIFTED`, where labels began to change. *Honest limit:* low temperature makes answers repeatable, not correct; and three runs per temperature is a small sample.

**3. "What is the difference between temperature and top_p?"**
*Model answer:* Temperature changes how spread out the probabilities are; top_p keeps only the smallest set of tokens whose probabilities add up to p. Teams usually adjust one and leave the other at its default. *In this lab:* `ask()` exposes only temperature; section 4's illustration showed a 12% option dropped by `top_p=0.8`. *Honest limit:* exact behaviour and allowed ranges vary by provider; check the provider's documentation.

**4. "Is temperature 0 fully deterministic?"**
*Model answer:* No. It is as close to "always pick the most likely token" as the provider allows, but hosted services can still vary slightly, for example because of how requests are batched and computed. *In this lab:* temperature 0 was usually stable across three runs; if anyone saw a change at 0, that is the reason. *Honest limit:* if you need exact reproducibility (for audit), store the inputs, the settings and the output of each decision, rather than relying on re-running the model.

**5. "What happens if max_tokens is too low, especially on a reasoning model?"**
*Model answer:* The reply is cut at the ceiling, often mid-sentence. A reasoning model's hidden thinking counts against the same budget, so a low limit can leave nothing visible. *In this lab:* `max_tokens=15` versus `150`; and an earlier `classify_ticket` with `max_tokens=10` returned empty replies (`INVALID:''`) on the course's reasoning model, which is why it now uses 1000. *Honest limit:* the right value depends on model and task; use it as a safety ceiling and ask for brevity in the prompt.

## C4. Troubleshooting

| Error or symptom | Cause | Fix |
|---|---|---|
| `Groq not configured yet: no real API_KEY in .env` | no `.env`, or the key is still `paste_your_groq_key_here` | `copy .env.example .env`, paste your key (no quotes, no trailing space), restart the kernel |
| `AuthenticationError` / `401` | wrong, expired or revoked key | create a new key in the provider's console, paste it, restart |
| `Gemini not configured yet: GEMINI_API_KEY is not set` | no Gemini key in `.env` | aistudio.google.com → Get API key → paste as `GEMINI_API_KEY`, restart |
| `using: None` | no plain Flash name listed, or a network problem | pin a listed name in `GEMINI_MODEL` |
| Gemini `404` / model not found | `GEMINI_MODEL` pinned to a retired name | leave `GEMINI_MODEL=` blank so section 1 picks a current one |
| `ollama: error (APIConnectionError ...)` | Ollama not running or not installed | optional today; start the Ollama app or run `ollama serve` |
| `ollama: error (NotFoundError ...)` | model not downloaded | `ollama pull llama3.2:3b` |
| `RateLimitError` / `429` | too many requests per minute (common on free tiers when a whole class runs the sweep) | wait 30–60 s; set `RUNS_PER_TEMPERATURE = 2` temporarily |
| `None`, empty text, or `INVALID:''` | the reasoning model used the whole `max_tokens` budget before any visible text | expected at very low limits; do not change the course numbers, report it |
| Groq `404` that mentions the model name with extra text | a comment was written on the same line as `MODEL=` | put comments on their own line; restart |
| Live Gemini test fails with an empty model name | `GEMINI_MODEL=` is blank and the test reads it as-is | pin the name section 1 printed after `using:` |

## C5. What comes next
**Session 12 — Local and open models:** the same client pointed at Ollama in depth, two local models compared, quantization, and when policy requires local inference. Keep:
- your working `.env` (Groq lines active, Gemini key in place); Session 12 compares a hosted model with a fully local one;
- Ollama installed with `llama3.2:3b` pulled, if you can;
- the habit from today: same `ask()`, only the client and model name change.

Before Session 12: notebook run top to bottom, `uv run pytest tests/test_s11.py -rs` green, reflections 1–4 written. Bring a short document (PDF or text) for Session 13.